In [28]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/processed/marketplace.db")
out_dir = Path("../data/processed/exports")
out_dir.mkdir(parents=True, exist_ok=True)

queries = {
    "kpi_marketplace_performance": "../sql/02_kpi_marketplace_performance.sql",
    "sku_deep_dive": "../sql/03_sku_brand_channel_deep_dive.sql",
    "operational_risk_metrics": "../sql/04_operational_risk_metrics.sql"
}

# --- Executive SKU Alerts (your existing logic, kept as-is) ---
alert_query = """
WITH monthly_metrics AS (
    SELECT
        StockCode,
        month,
        SUM(Quantity) AS units_sold,
        SUM(revenue) AS revenue
    FROM marketplace_orders
    GROUP BY StockCode, month
),
lagged AS (
    SELECT
        StockCode,
        month,
        units_sold,
        revenue,
        LAG(units_sold) OVER (PARTITION BY StockCode ORDER BY month) AS prev_units,
        LAG(revenue) OVER (PARTITION BY StockCode ORDER BY month) AS prev_revenue
    FROM monthly_metrics
)
SELECT
    StockCode,
    month,
    units_sold,
    revenue,
    ROUND((units_sold - prev_units) * 1.0 / prev_units, 2) AS unit_change_pct,
    ROUND((revenue - prev_revenue) * 1.0 / prev_revenue, 2) AS revenue_change_pct
FROM lagged
WHERE prev_units IS NOT NULL
  AND (
        units_sold < prev_units * 0.7
     OR revenue < prev_revenue * 0.7
  )
ORDER BY revenue_change_pct;
"""

conn = sqlite3.connect(db_path)

# Run exports
for name, sql_file in queries.items():
    with open(sql_file, "r", encoding="utf-8") as f:
        q = f.read()
    df = pd.read_sql_query(q, conn)
    df.to_csv(out_dir / f"{name}.csv", index=False)
    print(f" {name}: {df.shape} -> saved {out_dir / f'{name}.csv'}")

# Export alerts
alerts_df = pd.read_sql_query(alert_query, conn)
alerts_df.to_csv(out_dir / "executive_sku_alerts.csv", index=False)
print(f" executive_sku_alerts: {alerts_df.shape} -> saved {out_dir / 'executive_sku_alerts.csv'}")

# --- HARD CHECK: risk distribution MUST show 3 categories now ---
risk_df = pd.read_csv(out_dir / "operational_risk_metrics.csv")
print("\n Risk distribution (must NOT be only Low Risk):")
print(risk_df["risk_flag"].value_counts(dropna=False))

conn.close()
print("\n ALL EXPORTS SAVED TO:", out_dir.resolve())


 kpi_marketplace_performance: (1523, 8) -> saved ..\data\processed\exports\kpi_marketplace_performance.csv
 sku_deep_dive: (200, 8) -> saved ..\data\processed\exports\sku_deep_dive.csv
 operational_risk_metrics: (3665, 8) -> saved ..\data\processed\exports\operational_risk_metrics.csv
 executive_sku_alerts: (10122, 6) -> saved ..\data\processed\exports\executive_sku_alerts.csv

 Risk distribution (must NOT be only Low Risk):
risk_flag
Low Risk       1537
Medium Risk    1064
High Risk      1064
Name: count, dtype: int64

 ALL EXPORTS SAVED TO: C:\Users\shubh\marketplace-analytics-dashboard\data\processed\exports
